# 第 7 讲：并行训练

这是 CS336 Lecture 07 的可执行中文学习版。

- 官方来源：`external/cs336-lectures/lecture_07.py`
- 官方形式：executable lecture
- 英文标题：Parallelism

这份 notebook 是学习版，不是逐字翻译。它按官方讲义的结构和节奏整理主线，并在适合的位置加入小实验。


## 0. 本讲你要抓住什么

这一讲讲分布式训练的通信积木和训练并行：broadcast、reduce、all-reduce、all-gather、reduce-scatter，以及 data/tensor/pipeline parallelism。

学习方式：

1. 先读每节的中文解释。
2. 运行对应代码 cell。
3. 改一个参数，观察输出如何变化。
4. 最后用检查点问题自测。


## 1. 本讲路线图

先理解 collective communication，再把它们组合成分布式训练策略。


In [1]:
lecture = 7
title = 'Parallelism'
source = 'external/cs336-lectures/lecture_07.py'
print(f"Lecture {lecture}: {title}")
print("source:", source)


Lecture 7: Parallelism
source: external/cs336-lectures/lecture_07.py


## 2. 为什么需要并行

单卡显存、算力和训练时间都有限，大模型训练必须跨 GPU。


## 3. Collectives

all-reduce 合并梯度，all-gather 收集分片，reduce-scatter 合并并分发结果。


## 4. Data parallelism

每张卡有完整模型和不同 batch，反向后 all-reduce 梯度。


## 5. Tensor parallelism

把大矩阵或 attention head 切到多卡，单层内部就需要通信。


## 6. Pipeline parallelism

把层切到不同设备，microbatch 流水线执行，换取显存但带来 bubble。


## 动手实验 1：All-reduce 梯度玩具模拟

先直接运行，再改一个数字或字符串。


In [2]:
grads = [
    [1.0, 2.0, 3.0],
    [1.5, 2.5, 3.5],
    [0.5, 1.5, 2.5],
]
avg = [sum(values) / len(values) for values in zip(*grads)]
print("local grads:", grads)
print("all-reduced average:", avg)


local grads: [[1.0, 2.0, 3.0], [1.5, 2.5, 3.5], [0.5, 1.5, 2.5]]
all-reduced average: [1.0, 2.0, 3.0]


## 动手实验 2：All-gather 与 reduce-scatter 直觉

先直接运行，再改一个数字或字符串。


In [3]:
shards = [["p0"], ["p1"], ["p2"], ["p3"]]
gathered = sum(shards, [])
print("all-gather:", gathered)

values = [10, 20, 30, 40]
world_size = 4
reduced_then_scattered = [sum(values) / world_size]
print("reduce-scatter shard example:", reduced_then_scattered)


all-gather: ['p0', 'p1', 'p2', 'p3']
reduce-scatter shard example: [25.0]


## 动手实验 3：Pipeline bubble 粗略估计

先直接运行，再改一个数字或字符串。


In [4]:
def bubble_fraction(num_stages, microbatches):
    return (num_stages - 1) / (microbatches + num_stages - 1)

for microbatches in [1, 2, 4, 8, 16]:
    print(microbatches, bubble_fraction(4, microbatches))


1 0.75
2 0.6
4 0.42857142857142855
8 0.2727272727272727
16 0.15789473684210525


## 今日检查点

1. 能解释 all-reduce 在 data parallel 中做什么。
2. 能区分 data/tensor/pipeline parallel。
3. 能说出 pipeline bubble 为什么存在。

如果这些能讲清楚，这一讲的第一轮学习就完成了。
